# Calculus for Machine Learning

You don't need to be a calculus expert for ML. You need to understand one thing deeply:
**how to find the direction that makes a function decrease**. That's what training a model is.

This notebook covers only the calculus that matters: derivatives, gradients, and gradient descent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12

---
## 1. Derivatives — The Slope at a Point

The **derivative** of a function at a point tells you the **slope** of the curve at that exact spot.

- Positive slope → function is increasing (going uphill)
- Negative slope → function is decreasing (going downhill)
- Zero slope → you're at a peak, valley, or flat spot

**In ML:** The derivative tells the model **which direction to adjust** its parameters to reduce error.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

In [ ]:
def f(x):
    return x**3 - 3*x**2 + 2

def numerical_derivative(f, x, h=1e-7):
    return (f(x + h) - f(x - h)) / (2 * h)

x = np.linspace(-1, 3.5, 200)
points_of_interest = [0, 1, 2, 3]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(x, f(x), 'b-', lw=2, label='f(x) = x³ - 3x² + 2')

colors = ['red', 'green', 'orange', 'purple']
for xi, color in zip(points_of_interest, colors):
    slope = numerical_derivative(f, xi)
    tangent_x = np.linspace(xi - 0.8, xi + 0.8, 50)
    tangent_y = f(xi) + slope * (tangent_x - xi)

    ax.plot(tangent_x, tangent_y, '--', color=color, lw=2)
    ax.plot(xi, f(xi), 'o', color=color, markersize=8)
    ax.annotate(f'slope = {slope:.1f}', xy=(xi, f(xi)),
                xytext=(xi + 0.3, f(xi) + 0.8), fontsize=10, color=color,
                arrowprops=dict(arrowstyle='->', color=color))

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Derivative = Slope of Tangent Line at Each Point')
ax.legend()
plt.show()

---
## 2. Partial Derivatives — One Variable at a Time

When a function has **multiple inputs** (like a neural network with many weights),
a **partial derivative** tells you how the output changes when you tweak **just one** input,
keeping all others fixed.

For $f(x, y) = x^2 + 2y^2$:
- $\frac{\partial f}{\partial x} = 2x$ — how f changes as x changes
- $\frac{\partial f}{\partial y} = 4y$ — how f changes as y changes

In [ ]:
def f2d(x, y):
    return x**2 + 2*y**2

x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f2d(X, Y)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.7)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('f(x,y)')
ax1.set_title('f(x,y) = x² + 2y²')
ax1.view_init(elev=30, azim=45)

ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=15, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)

point = (2, 1)
df_dx = 2 * point[0]
df_dy = 4 * point[1]
ax2.plot(*point, 'ro', markersize=10)
ax2.annotate('', xy=(point[0] + df_dx*0.15, point[1] + df_dy*0.15), xytext=point,
             arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax2.text(point[0]+0.3, point[1]+0.8, f'∂f/∂x = {df_dx}\n∂f/∂y = {df_dy}', fontsize=11, color='red')

ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.set_title('Contour plot with partial derivatives at (2,1)')
ax2.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 3. The Chain Rule — Derivatives of Composed Functions

If you have a function inside a function: $h(x) = f(g(x))$, the chain rule says:

$$h'(x) = f'(g(x)) \cdot g'(x)$$

Multiply the derivatives along the chain.

**In ML:** Backpropagation IS the chain rule. A neural network is a chain of composed functions:

$$\text{output} = f_3(f_2(f_1(\text{input})))$$

To find how the loss changes with respect to an early weight, you multiply derivatives
all the way back through the chain.

In [ ]:
def g(x): return x**2 + 1
def f_outer(u): return np.sin(u)
def h(x): return f_outer(g(x))

def g_prime(x): return 2*x
def f_outer_prime(u): return np.cos(u)
def h_prime_chain(x): return f_outer_prime(g(x)) * g_prime(x)

x = np.linspace(-2, 2, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(x, g(x), 'b-', lw=2)
axes[0].set_title('Inner: g(x) = x² + 1')
axes[0].set_xlabel('x'); axes[0].set_ylabel('g(x)')

u = np.linspace(0, 6, 300)
axes[1].plot(u, f_outer(u), 'r-', lw=2)
axes[1].set_title('Outer: f(u) = sin(u)')
axes[1].set_xlabel('u'); axes[1].set_ylabel('f(u)')

axes[2].plot(x, h(x), 'g-', lw=2, label='h(x) = sin(x²+1)')
axes[2].plot(x, h_prime_chain(x), 'm--', lw=2, label="h'(x) = cos(x²+1)·2x")
axes[2].set_title('Composed: h(x) = f(g(x))')
axes[2].set_xlabel('x')
axes[2].legend()

plt.suptitle('Chain Rule: h\'(x) = f\'(g(x)) · g\'(x)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

x0 = 1.0
print(f"At x = {x0}:")
print(f"  g({x0}) = {g(x0)}")
print(f"  g'({x0}) = {g_prime(x0)}")
print(f"  f'(g({x0})) = f'({g(x0)}) = cos({g(x0)}) = {f_outer_prime(g(x0)):.4f}")
print(f"  Chain rule: h'({x0}) = {f_outer_prime(g(x0)):.4f} × {g_prime(x0)} = {h_prime_chain(x0):.4f}")
print(f"  Numerical check: {numerical_derivative(h, x0):.4f}")

### Backpropagation = Repeated Chain Rule

A simple neural network: `input → multiply by w1 → ReLU → multiply by w2 → loss`

To compute ∂loss/∂w1, you chain:

$$\frac{\partial \text{loss}}{\partial w_1} = \frac{\partial \text{loss}}{\partial \text{out}} \cdot \frac{\partial \text{out}}{\partial \text{hidden}} \cdot \frac{\partial \text{hidden}}{\partial w_1}$$

Each factor is a local derivative — multiply them all to get the gradient for the early weight.

---
## 4. Gradients — The Direction of Steepest Ascent

The **gradient** bundles all partial derivatives into a vector:

$$\nabla f(x, y) = \left[\frac{\partial f}{\partial x},\; \frac{\partial f}{\partial y}\right]$$

This vector **points in the direction of steepest increase**.
To decrease the function (minimize loss), go in the **opposite direction** — that's gradient descent!

In [ ]:
def grad_f2d(x, y):
    return np.array([2*x, 4*y])

x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f2d(X, Y)

fig, ax = plt.subplots(figsize=(8, 7))
contour = ax.contour(X, Y, Z, levels=15, cmap='coolwarm')
ax.clabel(contour, inline=True, fontsize=8)

grid_pts = np.linspace(-2.5, 2.5, 8)
for gx in grid_pts:
    for gy in grid_pts:
        grad = grad_f2d(gx, gy)
        grad_norm = np.linalg.norm(grad)
        if grad_norm > 0.3:
            grad_unit = grad / grad_norm * 0.3
            ax.annotate('', xy=(gx + grad_unit[0], gy + grad_unit[1]), xytext=(gx, gy),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Gradient Vectors on f(x,y) = x² + 2y²\nArrows point toward steepest ascent')
ax.set_aspect('equal')
plt.show()

---
## 5. Gradient Descent — The Core Algorithm of ML

**Gradient descent** is how ML models learn. The idea is dead simple:

1. Start at some random point
2. Compute the gradient (which direction is uphill)
3. Take a step in the **opposite** direction (downhill)
4. Repeat until you reach the bottom

**The update rule:**

$$w_{\text{new}} = w_{\text{old}} - \alpha \cdot \nabla f(w_{\text{old}})$$

where $\alpha$ is the **learning rate** — how big your steps are.

In [ ]:
def gradient_descent_1d(f, df, x0, lr=0.1, steps=20):
    path = [x0]
    x = x0
    for _ in range(steps):
        x = x - lr * df(x)
        path.append(x)
    return np.array(path)

loss = lambda x: (x - 3)**2 + 5
loss_grad = lambda x: 2*(x - 3)

path = gradient_descent_1d(loss, loss_grad, x0=-2, lr=0.2, steps=15)

x = np.linspace(-4, 8, 200)
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(x, loss(x), 'b-', lw=2, label='Loss: (x-3)² + 5')

for i in range(len(path) - 1):
    alpha = 1 - i / len(path)
    ax.plot(path[i], loss(path[i]), 'ro', markersize=8, alpha=alpha)
    ax.annotate('', xy=(path[i+1], loss(path[i+1])), xytext=(path[i], loss(path[i])),
                arrowprops=dict(arrowstyle='->', color='red', alpha=alpha, lw=1.5))

ax.plot(path[-1], loss(path[-1]), 'g*', markersize=20, label=f'Final: x = {path[-1]:.2f}')
ax.set_xlabel('x'); ax.set_ylabel('Loss')
ax.set_title('Gradient Descent: Rolling Downhill to the Minimum')
ax.legend(fontsize=11)
plt.show()

print(f"Started at x = {path[0]:.2f}, ended at x = {path[-1]:.4f}")
print(f"True minimum is at x = 3")

### 2D Gradient Descent — Descending a Surface

In [ ]:
def gradient_descent_2d(grad_fn, start, lr=0.1, steps=30):
    path = [np.array(start, dtype=float)]
    pos = np.array(start, dtype=float)
    for _ in range(steps):
        pos = pos - lr * grad_fn(pos[0], pos[1])
        path.append(pos.copy())
    return np.array(path)

path_2d = gradient_descent_2d(grad_f2d, start=[2.5, 2.5], lr=0.15, steps=25)

x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = f2d(X, Y)

fig, ax = plt.subplots(figsize=(8, 7))
contour = ax.contourf(X, Y, Z, levels=20, cmap='RdYlBu_r', alpha=0.6)
plt.colorbar(contour, ax=ax, label='f(x,y)')
ax.contour(X, Y, Z, levels=20, colors='gray', alpha=0.3)

ax.plot(path_2d[:, 0], path_2d[:, 1], 'ro-', markersize=5, lw=1.5, label='GD path')
ax.plot(path_2d[0, 0], path_2d[0, 1], 'rs', markersize=12, label='Start')
ax.plot(path_2d[-1, 0], path_2d[-1, 1], 'g*', markersize=15, label='End')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('2D Gradient Descent on f(x,y) = x² + 2y²')
ax.legend()
ax.set_aspect('equal')
plt.show()

---
## 6. Learning Rate — Too Big vs Too Small

The **learning rate** ($\alpha$) controls step size. Getting it right is critical:

- **Too large:** You overshoot the minimum, bouncing back and forth, possibly diverging
- **Too small:** You inch toward the minimum painfully slowly
- **Just right:** Quick, stable convergence

This is one of the most important **hyperparameters** in ML.

In [ ]:
learning_rates = [0.02, 0.2, 0.95]
labels = ['Too small (lr=0.02)', 'Just right (lr=0.2)', 'Too large (lr=0.95)']
colors = ['blue', 'green', 'red']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.linspace(-6, 8, 200)

for ax, lr, label, color in zip(axes, learning_rates, labels, colors):
    path = gradient_descent_1d(loss, loss_grad, x0=-2, lr=lr, steps=20)

    ax.plot(x, loss(x), 'b-', lw=2, alpha=0.5)
    for i in range(len(path) - 1):
        ax.plot(path[i], loss(path[i]), 'o', color=color, markersize=6)
    ax.plot(path[-1], loss(path[-1]), '*', color=color, markersize=15)

    ax.set_title(f'{label}\nFinal x = {path[-1]:.2f} (target: 3.00)', fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('Loss')
    ax.set_ylim(-5, 50)

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Convergence: Loss Over Steps

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for lr, label, color in zip([0.02, 0.2, 0.5], 
                             ['lr=0.02 (slow)', 'lr=0.2 (good)', 'lr=0.5 (fast)'],
                             ['blue', 'green', 'orange']):
    path = gradient_descent_1d(loss, loss_grad, x0=-2, lr=lr, steps=30)
    losses = [loss(xi) for xi in path]
    ax.plot(losses, '-o', color=color, markersize=4, label=label)

ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('Loss Curves for Different Learning Rates')
ax.legend()
plt.show()

---
## 7. Convexity — Why Shape Matters

A function is **convex** if it curves upward everywhere — like a bowl.
A convex function has **exactly one minimum**, so gradient descent is guaranteed to find it.

**Non-convex** functions (like neural network loss surfaces) have multiple local minima,
saddle points, and plateaus. Gradient descent might get stuck in a bad spot.

- Linear regression → convex loss ✓
- Logistic regression → convex loss ✓
- Neural networks → non-convex loss (but SGD works surprisingly well anyway)

In [ ]:
x = np.linspace(-3, 5, 300)

convex = lambda x: (x - 1)**2 + 2
non_convex = lambda x: x**4 - 5*x**2 + 4*x + 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x, convex(x), 'b-', lw=2)
ax.plot(1, convex(1), 'g*', markersize=15, label='Global minimum')
ax.set_title('Convex: One minimum\nGradient descent always finds it ✓')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend()

ax = axes[1]
ax.plot(x, non_convex(x), 'r-', lw=2)
x_mins = [-1.8, 1.3]
for xm in x_mins:
    ax.plot(xm, non_convex(xm), '*', markersize=15,
            color='green' if xm > 0 else 'orange')
ax.annotate('Local min', xy=(-1.8, non_convex(-1.8)),
            xytext=(-2.5, 8), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='orange'))
ax.annotate('Global min', xy=(1.3, non_convex(1.3)),
            xytext=(2.5, 8), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='green'))
ax.set_title('Non-convex: Multiple minima\nGD might get stuck in local min ✗')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')

plt.tight_layout()
plt.show()

### Gradient Descent on Non-Convex: Starting Point Matters

In [ ]:
non_convex_grad = lambda x: 4*x**3 - 10*x + 4

starts = [-3, -0.5, 2, 4]
colors_nc = ['blue', 'red', 'green', 'purple']

fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(-3, 4, 300)
ax.plot(x, non_convex(x), 'k-', lw=2, alpha=0.5)

for start, color in zip(starts, colors_nc):
    path = gradient_descent_1d(non_convex, non_convex_grad, x0=start, lr=0.01, steps=100)
    ax.plot(path, [non_convex(p) for p in path], 'o-', color=color,
            markersize=3, label=f'Start={start} → End={path[-1]:.2f}')
    ax.plot(start, non_convex(start), 's', color=color, markersize=10)

ax.set_title('Non-convex: Different starting points → different results')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend()
plt.show()

---
## 8. Full Implementation: Gradient Descent for Linear Regression

Let's put it all together. We'll implement gradient descent to fit a line to data.

**Model:** $y = wx + b$

**Loss (MSE):** $L = \frac{1}{n}\sum(y_i - (wx_i + b))^2$

**Gradients:**
- $\frac{\partial L}{\partial w} = \frac{-2}{n}\sum x_i(y_i - (wx_i + b))$
- $\frac{\partial L}{\partial b} = \frac{-2}{n}\sum (y_i - (wx_i + b))$

In [ ]:
np.random.seed(42)
X_data = np.random.rand(50) * 10
y_data = 2.5 * X_data + 7 + np.random.randn(50) * 3

w, b = 0.0, 0.0
lr = 0.005
n = len(X_data)
history = []

for epoch in range(200):
    y_pred = w * X_data + b
    mse = np.mean((y_data - y_pred)**2)
    history.append({'epoch': epoch, 'w': w, 'b': b, 'loss': mse})

    dw = (-2/n) * np.sum(X_data * (y_data - y_pred))
    db = (-2/n) * np.sum(y_data - y_pred)

    w = w - lr * dw
    b = b - lr * db

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(X_data, y_data, alpha=0.6, label='Data')
for snap in [history[0], history[10], history[50], history[-1]]:
    x_line = np.linspace(0, 10, 100)
    ax.plot(x_line, snap['w'] * x_line + snap['b'], '--',
            label=f"Epoch {snap['epoch']}: w={snap['w']:.1f}, b={snap['b']:.1f}")
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Line fit evolving over training')
ax.legend(fontsize=9)

ax = axes[1]
losses = [h['loss'] for h in history]
ax.plot(losses, 'b-', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Loss decreasing over training')

plt.tight_layout()
plt.show()

print(f"Final: w = {w:.4f} (true: 2.5), b = {b:.4f} (true: 7.0)")

---
## Summary: Calculus Concepts in ML

| Concept | Where It Shows Up |
|---------|------------------|
| **Derivative** | How loss changes with each parameter |
| **Partial derivative** | Gradient component for one weight |
| **Chain rule** | Backpropagation through layers |
| **Gradient** | Direction to update all weights |
| **Gradient descent** | THE training algorithm |
| **Learning rate** | How fast to update (hyperparameter) |
| **Convexity** | Determines if GD finds global optimum |